In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_community.chat_models.tongyi import ChatTongyi

from langgraph.cache import base
from openai import OpenAI
from dotenv import load_dotenv

import os

load_dotenv()

# 从环境提取 API 密钥和基础 URL
api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_API_URL")

# 初始化OpenAI 客户端
client = OpenAI(
    api_key=api_key,
    base_url=base_url
)

qwen = ChatOpenAI(
    api_key = api_key,
    base_url = base_url,
    model="qwen-max",
    extra_body={"enable_thinking": False} 
)

# 定义获取天气的函数
def get_weather(city: str) -> str:
   """ 获取天气信息
   """
   return f"今天{city}天气晴朗，气温35摄氏度，湿度10%，风速2米/秒，紫外线较强"

# 定义系统提示
system_prompt = "你是一个天气助手，你可以根据用户的问题，提供天气信息。"

agent = create_agent(
    model = qwen,
    tools = [get_weather],
    system_prompt =  system_prompt
)

result = agent.invoke(
    {
    "messages": [
        {"role": "user", "content": "西安天气"}
    ]
    }
)
print(result["messages"][-1].content_blocks)


[{'type': 'text', 'text': '今天西安的天气情况是晴朗，气温达到35摄氏度，湿度为10%，风速2米/秒，并且紫外线强度较高，请注意防晒。'}]


In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END

def mock_llm(state: MessagesState):
    return {"messages": [{"role": "ai", "content": "hello world"}]}

graph = StateGraph(MessagesState)
graph.add_node(mock_llm)
graph.add_edge(START, "mock_llm")
graph.add_edge("mock_llm", END)
graph = graph.compile()

graph.invoke({"messages": [{"role": "user", "content": "hi!"}]})

{'messages': [HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}, id='e7301ca8-f4c6-4ac8-9e78-71239cb18c5e'),
  AIMessage(content='hello world', additional_kwargs={}, response_metadata={}, id='933e6082-f0fb-4c2d-8971-edc17e922257', tool_calls=[], invalid_tool_calls=[])]}

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_API_URL")
# --------------------------
# 1. 配置 Qwen LLM（OpenAI兼容协议）
# --------------------------
llm = ChatOpenAI(
    api_key=api_key,
    base_url=base_url,
    model="qwen3.7-flash",
    extra_body={"enable_thinking": False},
    temperature=0
)

# --------------------------
# 2. 定义天气工具
# --------------------------
@tool
def get_weather(city: str) -> str:
    """
    查询指定城市天气
    Args:
        city: 城市名称
    """
    # 模拟天气接口，真实场景替换成http请求
    mock_data = {
        "北京": "晴天，26℃",
        "上海": "小雨，22℃",
        "西安": "多云，28℃"
    }
    return mock_data.get(city, f"暂无{city}天气数据")

tools = [get_weather]
    
# 把工具绑定到LLM，让模型知道可用工具
llm_with_tools = llm.bind_tools(tools)

# --------------------------
# 3. 定义节点函数
# --------------------------
def llm_node(state: MessagesState):
    """LLM节点：调用completion，可能输出工具调用"""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

def tool_node(state: MessagesState):
    """工具执行节点：解析tool_calls，执行get_weather"""
    messages = state["messages"]
    last_msg = messages[-1]
    tool_messages = []

    for tool_call in last_msg.tool_calls:
        tool_name = tool_call["name"]
        args = tool_call["args"]
        if tool_name == "get_weather":
            result = get_weather.invoke(args)
            tool_messages.append(ToolMessage(
                content=result,
                tool_call_id=tool_call["id"]
            ))
    return {"messages": tool_messages}

# --------------------------
# 4. 路由函数（条件分支核心）
# --------------------------
def route_after_llm(state: MessagesState):
    """判断下一步：调用工具 还是 直接结束"""
    last_msg = state["messages"][-1]
    # 如果存在tool_calls → 需要执行工具
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        return "need_tool"
    else:
        return "finish"

# --------------------------
# 5. 搭建Graph：add_node / add_edge / add_conditional_edges
# --------------------------
builder = StateGraph(MessagesState)

# 注册节点
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)

# 入口
builder.add_edge(START, "llm_node")

# 条件边：llm执行完进行分支判断
builder.add_conditional_edges(
    source="llm_node",
    path=route_after_llm,
    path_map={
        "need_tool": "tool_node",
        "finish": END
    }
)

# 无条件边：工具执行完成，回流到LLM，形成循环
builder.add_edge("tool_node", "llm_node")

# 编译图
graph = builder.compile()

# --------------------------
# 6. 运行测试
# --------------------------
if __name__ == "__main__":
    res = graph.invoke({
        "messages": [
            {"role": "user", "content": "南昌天气"}
            ]
        }
    )

    # 打印完整消息流
    for msg in res["messages"]:
        print(f"\n[{msg.type}] {msg.content}")


[human] 南昌天气

[ai] 

[tool] 暂无南昌天气数据

[ai] 对不起，暂时没有南昌的天气数据。您可以稍后再试，或者访问气象网站获取更多信息。
